In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy qiskit-addon-opt-mapper qiskit-ibm-catalog requests
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# แก้ปัญหา Market Split ด้วย Iskay Quantum Optimizer ของ Kipu Quantum

> **Note:** Qiskit Functions เป็นฟีเจอร์ทดลองที่ใช้ได้เฉพาะผู้ใช้ IBM Quantum&reg; Premium Plan, Flex Plan และ On-Prem (ผ่าน IBM Quantum Platform API) Plan เท่านั้น อยู่ในสถานะ preview release และอาจมีการเปลี่ยนแปลงได้

*ประมาณการใช้งาน: 20 วินาทีบนโปรเซสเซอร์ Heron r2 (หมายเหตุ: นี่เป็นการประมาณการเท่านั้น เวลาที่ใช้จริงอาจแตกต่างออกไป)*
## พื้นหลัง
บทช่วยสอนนี้สาธิตวิธีแก้ปัญหา Market Split โดยใช้ [Iskay quantum optimizer ของ Kipu Quantum](/guides/kipu-optimization) [\[1\]](#references) ปัญหา Market Split แทนความท้าทายด้านการจัดสรรทรัพยากรในโลกจริง ที่ต้องแบ่งตลาดออกเป็นเขตการขายที่สมดุลเพื่อให้ตรงกับเป้าหมายความต้องการที่กำหนด

### ความท้าทายของ Market Split
ปัญหา Market Split นำเสนอความท้าทายที่ดูเรียบง่ายแต่ซับซ้อนทางการคำนวณในด้านการจัดสรรทรัพยากร ลองนึกภาพบริษัทที่มีสินค้า $m$ รายการขายในตลาด $n$ แห่ง โดยแต่ละตลาดซื้อสินค้าเป็นชุด (แทนด้วยคอลัมน์ของเมทริกซ์ $A$) เป้าหมายทางธุรกิจคือการแบ่งตลาดเหล่านี้ออกเป็นสองเขตการขายที่สมดุล เพื่อให้แต่ละเขตได้รับความต้องการรวมสำหรับสินค้าทุกชิ้นพอดีครึ่งหนึ่ง

**สูตรคณิตศาสตร์:**

เราหาเวกเตอร์การกำหนด binary $x$ โดยที่:
- $x_j = 1$ กำหนดตลาด $j$ ให้เขต A
- $x_j = 0$ กำหนดตลาด $j$ ให้เขต B
- เงื่อนไข $Ax = b$ ต้องสอดคล้อง โดยที่ $b$ แทนเป้าหมายการขาย (โดยทั่วไปคือครึ่งหนึ่งของความต้องการรวมต่อสินค้า)

**ฟังก์ชันค่าใช้จ่าย:**

เพื่อแก้ปัญหานี้ เราย่อผลรวมกำลังสองของการละเมิดเงื่อนไข:

$$C(x) = ||Ax - b||^2 = \sum_{i=1}^{m} \left(\sum_{j=1}^{n} A_{ij}x_j - b_i\right)^2$$

โดยที่:
- $A_{ij}$ แทนยอดขายของสินค้า $i$ ในตลาด $j$
- $x_j \in {0,1}$ คือการกำหนด binary ของตลาด $j$
- $b_i$ คือเป้าหมายการขายสินค้า $i$ ในแต่ละเขต
- ค่าใช้จ่ายเท่ากับศูนย์พอดีเมื่อเงื่อนไขทั้งหมดสอดคล้อง

แต่ละพจน์ในผลรวมแทนกำลังสองของส่วนเบี่ยงเบนจากเป้าหมายการขายของสินค้าแต่ละชิ้น เมื่อขยายฟังก์ชันค่าใช้จ่ายนี้ จะได้:

$$C(x) = x^T A^T A x - 2b^T A x + b^T b$$

เนื่องจาก $b^T b$ เป็นค่าคงที่ การย่อ $C(x)$ จึงเทียบเท่ากับการย่อฟังก์ชันกำลังสอง $x^T A^T A x - 2b^T A x$ ซึ่งเป็นปัญหา QUBO (Quadratic Unconstrained Binary Optimization) พอดี

**ความซับซ้อนเชิงการคำนวณ:**

แม้จะมีการตีความทางธุรกิจที่ตรงไปตรงมา ปัญหานี้กลับมีความยากทางการคำนวณที่น่าทึ่ง:
- **ล้มเหลวในขนาดเล็ก**: ตัวแก้ Mixed Integer Programming แบบดั้งเดิมล้มเหลวกับอินสแตนซ์ที่มีสินค้าเพียงเจ็ดรายการภายใต้เวลาหมดอายุหนึ่งชั่วโมง [\[4\]](#references)
- **การเติบโตแบบเอกซ์โพเนนเชียล**: พื้นที่ผลเฉลยเติบโตแบบเอกซ์โพเนนเชียล ($2^n$ การกำหนดที่เป็นไปได้) ทำให้วิธีแบบ brute force ใช้ไม่ได้จริง

อุปสรรคการคำนวณที่รุนแรงนี้ ประกอบกับความเกี่ยวข้องกับการวางแผนเขตและการจัดสรรทรัพยากรในทางปฏิบัติ ทำให้ปัญหา Market Split เป็น benchmark ที่เหมาะสำหรับอัลกอริทึมออปติไมซ์ควอนตัม [\[4\]](#references)

### อะไรทำให้แนวทางของ Iskay แตกต่าง?
Iskay optimizer ใช้อัลกอริทึม **bf-DCQO (bias-field digitized counterdiabatic quantum optimization)** [\[1\]](#references) ซึ่งถือเป็นความก้าวหน้าที่สำคัญในการออปติไมซ์ควอนตัม:

**ประสิทธิภาพของ Circuit**: อัลกอริทึม bf-DCQO บรรลุการลด Gate ที่น่าทึ่ง [\[1\]](#references):
- ใช้ **entangling gates น้อยกว่าถึง 10 เท่า** เมื่อเทียบกับ Digital Quantum Annealing (DQA)
- Circuit ที่ตื้นกว่าอย่างมีนัยสำคัญช่วยให้:
  - ลดการสะสมข้อผิดพลาดระหว่างการประมวลผลควอนตัม
  - สามารถรับมือกับปัญหาขนาดใหญ่กว่าบนฮาร์ดแวร์ควอนตัมปัจจุบัน
  - ไม่ต้องใช้เทคนิคการลดข้อผิดพลาด

**การออกแบบแบบ non-variational**: ต่างจากอัลกอริทึม variational ที่ต้องใช้ประมาณ 100 รอบซ้ำ bf-DCQO โดยทั่วไปต้องใช้เพียง **ประมาณ 10 รอบซ้ำ** [\[1\]](#references) ซึ่งทำได้ผ่าน:
- การคำนวณ bias-field อย่างชาญฉลาดจากการกระจายสถานะที่วัดได้
- การเริ่มต้นแต่ละรอบซ้ำจากสถานะพลังงานใกล้กับผลเฉลยก่อนหน้า
- การประมวลผลคลาสสิกแบบรวมกับการค้นหาเฉพาะที่

**โปรโตคอล Counterdiabatic**: อัลกอริทึมรวมพจน์ counterdiabatic ที่ยับยั้งการกระตุ้นควอนตัมที่ไม่ต้องการในช่วงเวลาวิวัฒนาการสั้น ทำให้ระบบอยู่ใกล้สถานะพื้นแม้จะมีการเปลี่ยนแปลงอย่างรวดเร็ว [\[1\]](#references)
## ข้อกำหนดเบื้องต้น
ก่อนเริ่มบทช่วยสอนนี้ ต้องติดตั้งสิ่งต่อไปนี้:

* Qiskit IBM Runtime (`pip install qiskit-ibm-runtime`)
* Qiskit Functions (`pip install qiskit-ibm-catalog`)
* NumPy (`pip install numpy`)
* Requests (`pip install requests`)
* Opt Mapper Qiskit addon (`pip install qiskit-addon-opt-mapper`)

นอกจากนี้ยังต้องขอสิทธิ์เข้าถึง [ฟังก์ชัน Iskay Quantum Optimizer](https://quantum.cloud.ibm.com/functions?id=kipu-quantum-iskay-quantum-optimizer) จาก Qiskit Functions Catalog
## การตั้งค่า
อันดับแรก นำเข้าแพ็กเกจที่จำเป็นทั้งหมดสำหรับบทช่วยสอนนี้

In [ ]:
import os
import tempfile
import time
from typing import Tuple, Optional

import numpy as np
import requests

from qiskit_ibm_catalog import QiskitFunctionsCatalog

from qiskit_addon_opt_mapper import OptimizationProblem
from qiskit_addon_opt_mapper.converters import OptimizationProblemToQubo

print("All required libraries imported successfully")

### กำหนดค่า IBM Quantum credentials
กำหนด credentials ของ [IBM Quantum&reg; Platform](https://quantum.cloud.ibm.com/) โดยต้องใช้:
- **API Token**: API key 44 ตัวอักษรจาก IBM Quantum Platform
- **Instance CRN**: ตัวระบุ IBM Cloud&reg; instance ของคุณ

In [ ]:
token = "<YOUR_API_KEY>"
instance = "<YOUR_INSTANCE_CRN>"

## ขั้นตอนที่ 1: แมป input คลาสสิกไปยังปัญหาควอนตัม
เราเริ่มต้นด้วยการแมปปัญหาคลาสสิกไปยังรูปแบบที่เข้ากันได้กับควอนตัม ขั้นตอนนี้ประกอบด้วย:

1. เชื่อมต่อกับ Iskay Quantum Optimizer
2. โหลดและกำหนดปัญหา Market Split
3. ทำความเข้าใจอัลกอริทึม bf-DCQO ที่จะแก้ปัญหา

### เชื่อมต่อกับ Iskay Quantum Optimizer
เราเริ่มด้วยการสร้างการเชื่อมต่อกับ Qiskit Functions Catalog และโหลด Iskay Quantum Optimizer Iskay Optimizer เป็นฟังก์ชันควอนตัมที่จัดทำโดย Kipu Quantum ซึ่ง implement อัลกอริทึม bf-DCQO สำหรับแก้ปัญหาออปติไมซ์บนฮาร์ดแวร์ควอนตัม

In [ ]:
catalog = QiskitFunctionsCatalog(token=token, instance=instance)
iskay_solver = catalog.load("kipu-quantum/iskay-quantum-optimizer")

print("Iskay optimizer loaded successfully")
print("Ready to solve optimization problems using bf-DCQO algorithm")

### โหลดและกำหนดปัญหา

#### ทำความเข้าใจรูปแบบข้อมูลปัญหา

อินสแตนซ์ปัญหาจาก QOBLIB (Quantum Optimization Benchmarking Library) [\[2\]](#references) ถูกเก็บในรูปแบบไฟล์ข้อความง่าย ๆ มาดูเนื้อหาจริงของอินสแตนซ์เป้าหมาย `ms_03_200_177.dat`:

In [ ]:
def parse_marketsplit_dat(filename: str) -> Tuple[np.ndarray, np.ndarray]:
    """
    Parse a market split problem from a .dat file format.

    Parameters
    ----------
    filename : str
        Path to the .dat file containing the market split problem data.

    Returns
    -------
    A : np.ndarray
        Coefficient matrix of shape (m, n) where m is the number of products
        and n is the number of markets.
    b : np.ndarray
        Target vector of shape (m,) containing the target sales per product.
    """
    with open(filename, "r", encoding="utf-8") as f:
        lines = [
            line.strip()
            for line in f
            if line.strip() and not line.startswith("#")
        ]

    if not lines:
        raise ValueError("Empty or invalid .dat file")

    # First line: m n (number of products and markets)
    m, n = map(int, lines[0].split())

    # Next m lines: each row of A followed by corresponding element of b
    A, b = [], []
    for i in range(1, m + 1):
        values = list(map(int, lines[i].split()))
        A.append(values[:-1])  # First n values: product sales per market
        b.append(values[-1])  # Last value: target sales for this product

    return np.array(A, dtype=np.int32), np.array(b, dtype=np.int32)


def fetch_marketsplit_data(
    instance_name: str = "ms_03_200_177.dat",
) -> Tuple[Optional[np.ndarray], Optional[np.ndarray]]:
    """
    Fetch market split data directly from the QOBLIB repository.

    Parameters
    ----------
    instance_name : str
        Name of the .dat file to fetch (default: "ms_03_200_177.dat").

    Returns
    -------
    A : np.ndarray or None
        Coefficient matrix if successful, None if failed.
    b : np.ndarray or None
        Target vector if successful, None if failed.
    """
    url = f"https://git.zib.de/qopt/qoblib-quantum-optimization-benchmarking-library/-/raw/main/01-marketsplit/instances/{instance_name}"

    try:
        response = requests.get(url, timeout=30)
        response.raise_for_status()

        with tempfile.NamedTemporaryFile(
            mode="w", suffix=".dat", delete=False, encoding="utf-8"
        ) as f:
            f.write(response.text)
            temp_path = f.name

        try:
            return parse_marketsplit_dat(temp_path)
        finally:
            os.unlink(temp_path)
    except Exception as e:
        print(f"Error: {e}")
        return None, None

**โครงสร้างรูปแบบ:**
- **บรรทัดแรก:** `3 20`
  - `3` = จำนวนสินค้า (เงื่อนไข/แถวในเมทริกซ์ $A$)
  - `20` = จำนวนตลาด (ตัวแปร/คอลัมน์ในเมทริกซ์ $A$)

- **3 บรรทัดถัดมา:** เมทริกซ์ coefficient $A$ และเวกเตอร์เป้าหมาย $b$
  - แต่ละบรรทัดมี 21 ตัวเลข: 20 ตัวแรกเป็น row coefficients ตัวสุดท้ายเป็นเป้าหมาย
  - บรรทัด 2: `60 92 161 ... 51 | 1002`
    - 20 ตัวเลขแรก: ยอดขายสินค้า 1 ในแต่ละตลาด 20 แห่ง
    - ตัวเลขสุดท้าย (1002): เป้าหมายการขายสินค้า 1 ในแต่ละเขต
  - บรรทัด 3: `176 196 41 ... 46 | 879`
    - ยอดขายสินค้า 2 ต่อตลาดและเป้าหมาย (879)
  - บรรทัด 4: `68 68 179 ... 95 | 1040`
    - ยอดขายสินค้า 3 ต่อตลาดและเป้าหมาย (1040)

**การตีความทางธุรกิจ:**
- ตลาด 0 ขาย: สินค้า 1 จำนวน 60 หน่วย, สินค้า 2 จำนวน 176 หน่วย, สินค้า 3 จำนวน 68 หน่วย
- ตลาด 1 ขาย: สินค้า 1 จำนวน 92 หน่วย, สินค้า 2 จำนวน 196 หน่วย, สินค้า 3 จำนวน 68 หน่วย
- และต่อไปสำหรับตลาดทั้ง 20 แห่ง...
- **เป้าหมาย**: แบ่งตลาด 20 แห่งนี้ออกเป็นสองเขต โดยแต่ละเขตได้รับสินค้า 1 พอดี 1002 หน่วย, สินค้า 2 พอดี 879 หน่วย และสินค้า 3 พอดี 1040 หน่วย

#### การแปลงเป็น QUBO
## จากเงื่อนไขสู่ QUBO: การแปลงทางคณิตศาสตร์
พลังของการออปติไมซ์ควอนตัมอยู่ที่การแปลงปัญหาแบบมีเงื่อนไขเป็นรูปแบบ quadratic แบบไม่มีเงื่อนไข [\[4\]](#references) สำหรับปัญหา Market Split เราแปลงเงื่อนไขความเท่ากัน

$$ Ax = b $$

โดยที่ $x ∈ {0,1}^n$, เป็น QUBO โดยการลงโทษการละเมิดเงื่อนไข

**วิธีการ penalty:**
เนื่องจากเราต้องการให้ $Ax = b$ สอดคล้องอย่างแน่นอน เราจึงย่อผลรวมกำลังสองของการละเมิด:
$$f(x) = ||Ax - b||^2$$

ซึ่งเท่ากับศูนย์พอดีเมื่อเงื่อนไขทั้งหมดสอดคล้อง การขยายทางพีชคณิต:
$$f(x) = (Ax - b)^T(Ax - b) = x^T A^T A x - 2b^T A x + b^T b$$

**เป้าหมาย QUBO:**
เนื่องจาก $b^T b$ เป็นค่าคงที่ การออปติไมซ์ของเราจึงกลายเป็น:
$$\text{minimize} \quad Q(x) = x^T(A^T A)x - 2(A^T b)^T x$$

**ข้อคิดสำคัญ:** การแปลงนี้แม่นยำ ไม่ใช่การประมาณ เงื่อนไขความเท่ากันกลายเป็นรูปแบบ quadratic โดยธรรมชาติผ่านการยกกำลังสอง โดยไม่ต้องใช้ตัวแปรเสริมหรือพารามิเตอร์ penalty ทำให้การกำหนดสูตรนี้มีความงดงามทางคณิตศาสตร์และมีประสิทธิภาพเชิงการคำนวณสำหรับตัวแก้ควอนตัม [\[4\]](#references) เราจะใช้คลาส `OptimizationProblem` เพื่อกำหนดปัญหาแบบมีเงื่อนไข จากนั้นแปลงเป็นรูปแบบ QUBO โดยใช้ `OptimizationProblemToQubo` ซึ่งทั้งคู่มาจากแพ็กเกจ **qiskit_addon_opt_mapper** ซึ่งจัดการการแปลงแบบ penalty-based โดยอัตโนมัติ
### implement ฟังก์ชันโหลดข้อมูลและแปลง QUBO
ตอนนี้เรากำหนดฟังก์ชันอรรถประโยชน์สามตัว:
1. `parse_marketsplit_dat()` - แยกวิเคราะห์รูปแบบไฟล์ `.dat` และดึงเมทริกซ์ $A$ และ $b$
2. `fetch_marketsplit_data()` - ดาวน์โหลดอินสแตนซ์ปัญหาโดยตรงจาก repository QOBLIB

In [ ]:
# Load the problem instance
instance_name = "ms_03_200_177.dat"
A, b = fetch_marketsplit_data(instance_name=instance_name)

if A is not None:
    print("Successfully loaded problem instance from QOBLIB")
    print("\nProblem Instance Analysis:")
    print("=" * 50)
    print(f"Coefficient Matrix A: {A.shape[0]} × {A.shape[1]}")
    print(f"   → {A.shape[0]} products (constraints)")
    print(f"   → {A.shape[1]} markets (decision variables)")
    print(f"Target Vector b: {b}")
    print("   → Target sales per product for each region")
    print(
        f"Solution Space: 2^{A.shape[1]} = {2**A.shape[1]:,} possible assignments"
    )

### โหลดอินสแตนซ์ปัญหา
ตอนนี้เราโหลดอินสแตนซ์ปัญหาเฉพาะ `ms_03_200_177.dat` จาก QOBLIB [2] อินสแตนซ์นี้มี:
- 3 สินค้า (เงื่อนไข)
- 20 ตลาด (ตัวแปรการตัดสินใจ binary)
- การกำหนดตลาดที่เป็นไปได้มากกว่า 1 ล้านแบบ ($2^{20} = 1,048,576$)

In [ ]:
# Create optimization problem
ms = OptimizationProblem(instance_name.replace(".dat", ""))

# Add binary variables (one for each market)
ms.binary_var_list(A.shape[1])

# Add equality constraints (one for each product)
for idx, rhs in enumerate(b):
    ms.linear_constraint(A[idx, :], sense="==", rhs=rhs)

# Convert to QUBO with penalty parameter
qubo = OptimizationProblemToQubo(penalty=1).convert(ms)

print("QUBO Conversion Complete:")
print("=" * 50)
print(f"Number of variables: {qubo.get_num_vars()}")
print(f"Constant term: {qubo.objective.constant}")
print(f"Linear terms: {len(qubo.objective.linear.to_dict())}")
print(f"Quadratic terms: {len(qubo.objective.quadratic.to_dict())}")

### แปลงเป็นรูปแบบ QUBO
ตอนนี้เราแปลงปัญหาออปติไมซ์แบบมีเงื่อนไขเป็นรูปแบบ QUBO:

In [ ]:
# Convert QUBO to Iskay dictionary format:

# Create empty Iskay input dictionary
iskay_input_problem = {}

# Convert QUBO to Iskay dictionary format
iskay_input_problem = {"()": qubo.objective.constant}

for i in range(qubo.get_num_vars()):
    for j in range(i, qubo.get_num_vars()):
        if i == j:
            # Add linear term (including diagonal quadratic contribution)
            iskay_input_problem[f"({i}, )"] = float(
                qubo.objective.linear.to_dict().get(i)
            ) + float(qubo.objective.quadratic.to_dict().get((i, i)))
        else:
            # Add off-diagonal quadratic term
            iskay_input_problem[f"({i}, {j})"] = float(
                qubo.objective.quadratic.to_dict().get((i, j))
            )

# Display Iskay dictionary summary
print("Iskay Dictionary Format:")
print("=" * 50)
print(f"Total coefficients: {len(iskay_input_problem)}")
print(f"  • Constant term: {iskay_input_problem['()']}")
print(
    f"  • Linear terms: {sum(1 for k in iskay_input_problem.keys() if k != '()' and ', )' in k)}"
)
print(
    f"  • Quadratic terms: {sum(1 for k in iskay_input_problem.keys() if k != '()' and ', )' not in k)}"
)
print("\nSample coefficients:")

# Get first 10 and last 5 items properly
items = list(iskay_input_problem.items())
first_10 = list(enumerate(items[:10]))
last_5 = list(enumerate(items[-5:], start=len(items) - 5))

for i, (key, value) in first_10 + last_5:
    coeff_type = (
        "constant"
        if key == "()"
        else "linear"
        if ", )" in key
        else "quadratic"
    )
    print(f"  {key}: {value} ({coeff_type})")
print("  ...")
print("\n✓ Problem ready for Iskay optimizer!")

### แปลง QUBO เป็นรูปแบบ Iskay
ตอนนี้เราต้องแปลงออบเจ็กต์ QUBO เป็นรูปแบบ dictionary ที่ Iskay Optimizer ของ Kipu Quantum ต้องการ

อาร์กิวเมนต์ `problem` และ `problem_type` เข้ารหัสปัญหาออปติไมซ์ในรูปแบบ

$$
\begin{align}
\min_{(x_1, x_2, \ldots, x_n) \in D} C(x_1, x_2, \ldots, x_n) \nonumber
\end{align}
$$
โดยที่

$$
C(x_1, ... , x_n) = a + \sum_{i} b_i x_i + \sum_{i, j} c_{i, j} x_i x_j + ... + \sum_{k_1, ..., k_m} g_{k_1, ..., k_m} x_{k_1} ... x_{k_m}
$$

- การเลือก `problem_type = "binary"` หมายความว่าฟังก์ชันค่าใช้จ่ายอยู่ในรูปแบบ `binary` ซึ่งหมายความว่า $D = {0,  1}^{n}$ นั่นคือฟังก์ชันค่าใช้จ่ายเขียนในรูปแบบ QUBO/HUBO
- ในทางกลับกัน การเลือก `problem_type = "spin"` หมายความว่าฟังก์ชันค่าใช้จ่ายเขียนในรูปแบบ Ising โดยที่ $D = {-1, 1}^{n}$

coefficients ของปัญหาควรเข้ารหัสใน dictionary ดังนี้:
$$
\begin{align} \nonumber
&\texttt{{} \\ \nonumber
&\texttt{"()"}&: \quad &a, \\ \nonumber
&\texttt{"(i,)"}&: \quad &b_i, \\ \nonumber
&\texttt{"(i, j)"}&: \quad &c_{i, j}, \quad (i \neq j) \\ \nonumber
&\quad  \vdots \\ \nonumber
&\texttt{"(} k_1, ..., k_m  \texttt{)"}&: \quad &g_{k_1, ..., k_m}, \quad (k_1 \neq k_2 \neq \dots \neq k_m) \\ \nonumber
&\texttt{}}
\end{align}
$$

โปรดสังเกตว่า keys ของ dictionary ต้องเป็น string ที่ประกอบด้วย tuple ของจำนวนเต็มที่ไม่ซ้ำกัน สำหรับปัญหา binary เราทราบว่า:

$$
x_i^2 = x_i
$$

สำหรับ $i=j$ (เนื่องจาก $x_i \in {0,1}$ หมายความว่า $x_i \cdot x_i = x_i$) ดังนั้นในรูปแบบ QUBO ของคุณ หากมีทั้ง linear contributions $b_i x_i$ และ diagonal quadratic contributions $c_{i,i} x_i^2$ พจน์เหล่านี้ต้องรวมเป็น linear coefficient เดียว:

**linear coefficient รวมสำหรับตัวแปร $x_i$:** $b_i + c_{i,i}$

ซึ่งหมายความว่า:
- Linear terms อย่าง `"(i, )"` ประกอบด้วย: linear coefficient ดั้งเดิม + diagonal quadratic coefficient
- Diagonal quadratic terms อย่าง `"(i, i)"` **ไม่ควร** ปรากฏใน dictionary สุดท้าย
- เฉพาะ off-diagonal quadratic terms อย่าง `"(i, j)"` ที่ $i \neq j$ เท่านั้นที่ควรรวมเป็น entries แยกต่างหาก

**ตัวอย่าง:** หาก QUBO มี $3x_1 + 2x_1^2 + 4x_1 x_2$ Iskay dictionary ควรประกอบด้วย:
- `"(0, )"`: `5.0` (รวม $3 + 2 = 5$)
- `"(0, 1)"`: `4.0` (off-diagonal term)

**ไม่ใช่** entries แยกสำหรับ `"(0, )"`: `3.0` และ `"(0, 0)"`: `2.0`

In [ ]:
# Specify the target backend
backend_name = "ibm_fez"

# Set the number of bias-field iterations and set a tag to identify the jobs
options = {
    "num_iterations": 3,  # Change number of bias-field iterations
    "job_tags": ["market_split_example"],  # Tag to identify jobs
}

# Configure Iskay optimizer
iskay_input = {
    "problem": iskay_input_problem,
    "problem_type": "binary",
    "backend_name": backend_name,
    "options": options,
}

print("Iskay Optimizer Configuration:")
print("=" * 40)
print(f"  Backend: {backend_name}")
print(f"  Problem: {len(iskay_input['problem'])} terms")
print("  Algorithm: bf-DCQO")

### ทำความเข้าใจอัลกอริทึม bf-DCQO
ก่อนที่จะรันการ optimization มาทำความเข้าใจอัลกอริทึม quantum ที่ซับซ้อนซึ่งเป็นพลังขับเคลื่อนของ Iskay กันก่อน: **bf-DCQO (bias-field digitized counterdiabatic quantum optimization)** [\[1\]](#references)

#### bf-DCQO คืออะไร
bf-DCQO อิงกับการวิวัฒนาการตามเวลาของระบบ quantum โดยคำตอบของปัญหาถูกเข้ารหัสไว้ใน **ground state** (สถานะพลังงานต่ำสุด) ของ Hamiltonian quantum สุดท้าย [\[1\]](#references) อัลกอริทึมนี้จัดการกับความท้าทายพื้นฐานใน quantum optimization:

**ความท้าทาย**: quantum computing แบบ adiabatic แบบดั้งเดิมต้องการการวิวัฒนาการที่ช้ามากเพื่อรักษาเงื่อนไข ground state ตาม adiabatic theorem ซึ่งต้องใช้ Circuit quantum ที่ลึกขึ้นเรื่อยๆ เมื่อความซับซ้อนของปัญหาเพิ่มขึ้น นำไปสู่การทำงานของ Gate มากขึ้นและข้อผิดพลาดสะสม

**วิธีแก้ปัญหา**: bf-DCQO ใช้ counterdiabatic protocol เพื่อให้วิวัฒนาการอย่างรวดเร็วในขณะที่รักษา ground state fidelity ไว้ได้ ลด circuit depth ได้อย่างมาก

#### กรอบทางคณิตศาสตร์
อัลกอริทึมนี้ minimize cost function ในรูปแบบ:

$$\min_{(x_1,x_2,...,x_n) \in D} C(x_1,x_2,...,x_n)$$

โดยที่ $D = {0,1}^n$ สำหรับตัวแปร binary และ:

$$C(x) = a + \sum_i b_i x_i + \sum_{i,j} c_{ij} x_i x_j + ... + \sum g_{k_1,...,k_m} x_{k_1}...x_{k_m}$$

สำหรับปัญหา Market Split ของเรา cost function คือ:

$$C(x) = ||Ax - b||^2 = x^T A^T A x - 2 b^T A x + b^T b$$

#### บทบาทของ counterdiabatic terms
**Counterdiabatic terms** คือ terms เพิ่มเติมที่ถูกนำเข้ามาใน Hamiltonian ที่ขึ้นกับเวลา เพื่อระงับการกระตุ้นที่ไม่ต้องการระหว่างการวิวัฒนาการ quantum นี่คือเหตุผลที่มันสำคัญมาก:

ใน quantum optimization แบบ adiabatic เราวิวัฒนาระบบตาม Hamiltonian ที่ขึ้นกับเวลา:

$$H(t) = \left(1 - \frac{t}{T}\right) H_{\text{initial}} + \frac{t}{T} H_{\text{problem}}$$

โดยที่ $H_{\text{problem}}$ เข้ารหัสปัญหา optimization ของเรา เพื่อรักษา ground state ระหว่างการวิวัฒนาการอย่างรวดเร็ว เราเพิ่ม counterdiabatic terms:

$$H_{\text{CD}}(t) = H(t) + H_{\text{counter}}(t)$$

counterdiabatic terms เหล่านี้ทำสิ่งต่อไปนี้:
1. **ระงับการเปลี่ยนแปลงที่ไม่ต้องการ**: ป้องกันไม่ให้ quantum state กระโดดไปยัง excited states ระหว่างการวิวัฒนาการอย่างรวดเร็ว
2. **ช่วยให้วิวัฒนาการในเวลาที่สั้นลง**: ทำให้เราไปถึงสถานะสุดท้ายได้เร็วกว่ามากโดยไม่ละเมิด adiabaticity
3. **ลด circuit depth**: การวิวัฒนาการที่สั้นลงนำไปสู่ Gate น้อยลงและข้อผิดพลาดน้อยลง

ผลกระทบในทางปฏิบัติมีความโดดเด่นมาก: bf-DCQO ใช้ **entangling Gate น้อยกว่าถึง 10 เท่า** เมื่อเทียบกับ Digital Quantum Annealing [\[1\]](#references) ทำให้สามารถใช้งานได้จริงบนฮาร์ดแวร์ quantum ที่มีสัญญาณรบกวนในปัจจุบัน

#### การ optimization แบบ bias-field iterative
ต่างจากอัลกอริทึม variational ที่ optimize พารามิเตอร์ Circuit ผ่านหลาย iteration, bf-DCQO ใช้ **แนวทางที่นำทางด้วย bias-field** ที่ converge ใน approximately 10 iterations [1]:

**กระบวนการ iteration:**

1. **การวิวัฒนาการ quantum เริ่มต้น**: เริ่มด้วย Circuit quantum ที่ implement counterdiabatic evolution protocol

2. **การวัดผล**: วัด quantum state เพื่อได้การแจกแจงความน่าจะเป็นบน bitstrings

3. **การคำนวณ bias-field**: วิเคราะห์สถิติการวัดและคำนวณ bias-field ที่เหมาะสมที่สุด $h_i$ สำหรับแต่ละ Qubit:
   $$h_i = \text{f}(\text{measurement statistics}, \text{previous solutions})$$

4. **Iteration ถัดไป**: bias-field แก้ไข Hamiltonian สำหรับ iteration ถัดไป:
   $$H_{\text{next}} = H_{\text{problem}} + \sum_i h_i \sigma_i^z$$

   ซึ่งช่วยให้เริ่มต้นใกล้กับ solution ที่ดีที่พบก่อนหน้า ทำให้เกิดรูปแบบหนึ่งของ "quantum local search"

5. **Convergence**: ทำซ้ำจนกว่าคุณภาพของ solution จะมีเสถียรภาพหรือถึงจำนวน iteration สูงสุด

**ข้อได้เปรียบสำคัญ**: แต่ละ iteration ให้ความก้าวหน้าที่มีความหมายต่อ solution ที่ดีที่สุดโดยการนำข้อมูลจากการวัดก่อนหน้ามาผนวก ต่างจากวิธี variational ที่ต้องสำรวจพื้นที่พารามิเตอร์โดยไม่มีข้อมูลนำ

#### การ post-processing แบบ classical ที่ผสมผสาน
หลังจาก quantum optimization converge แล้ว Iskay จะทำการ **local search** แบบ classical เพิ่มเติม:

- **การสำรวจแบบ bit-flip**: พลิก bit ใน solution ที่วัดได้ดีที่สุดอย่างเป็นระบบหรือแบบสุ่ม
- **การประเมิน energy**: คำนวณ $C(x)$ สำหรับแต่ละ solution ที่ถูกแก้ไข
- **การเลือกแบบ greedy**: ยอมรับการปรับปรุงที่ลด cost function
- **Multiple passes**: ทำหลาย pass (ควบคุมโดย `postprocessing_level`)

แนวทาง hybrid นี้ชดเชยข้อผิดพลาด bit-flip จากความไม่สมบูรณ์ของฮาร์ดแวร์และ readout errors เพื่อให้มั่นใจว่าได้ solution คุณภาพสูงแม้บนอุปกรณ์ quantum ที่มีสัญญาณรบกวน

#### เหตุใด bf-DCQO จึงโดดเด่นบนฮาร์ดแวร์ปัจจุบัน
อัลกอริทึม bf-DCQO ถูกออกแบบมาโดยเฉพาะเพื่อโดดเด่นบนอุปกรณ์ noisy intermediate-scale quantum (NISQ) ในปัจจุบัน [\[1\]](#references):

1. **ความทนทานต่อข้อผิดพลาด**: Gate น้อยลง (ลด 10 เท่า) หมายความว่าการสะสมข้อผิดพลาดลดลงอย่างมาก
2. **ไม่ต้องการ error mitigation**: ประสิทธิภาพโดยธรรมชาติของอัลกอริทึมขจัดความจำเป็นในการใช้เทคนิค error mitigation ที่มีค่าใช้จ่ายสูง [\[1\]](#references)
3. **Scalability**: รองรับปัญหาที่มี Qubit ได้ถึง 156 Qubit (ตัวแปร binary 156 ตัว) ด้วยการ mapping Qubit โดยตรง [\[1\]](#references)
4. **ประสิทธิภาพที่พิสูจน์แล้ว**: บรรลุ approximation ratio 100% บน benchmark MaxCut และ HUBO instances [\[1\]](#references)

มาดูอัลกอริทึมอันทรงพลังนี้ในการทำงานจริงกับปัญหา Market Split ของเรากัน!

## Step 2: Optimize problem for quantum hardware execution
อัลกอริทึม bf-DCQO จัดการ circuit optimization โดยอัตโนมัติ สร้าง Circuit quantum ตื้นพร้อม counterdiabatic terms ที่ออกแบบมาโดยเฉพาะสำหรับ Backend เป้าหมาย

### Configure the optimization
Iskay Optimizer ต้องการพารามิเตอร์สำคัญหลายตัวเพื่อแก้ปัญหา optimization ของคุณได้อย่างมีประสิทธิภาพ มาดูแต่ละพารามิเตอร์และบทบาทของมันใน quantum optimization กัน:

#### พารามิเตอร์ที่จำเป็น
| พารามิเตอร์ | ประเภท | คำอธิบาย | ตัวอย่าง |
|-----------|------|-------------|---------|
| **problem** | `Dict[str, float]` | QUBO coefficients ในรูปแบบ string-key | `{"()": -21.0, "(0,4)": 0.5, "(0,1)": 0.5}` |
| **problem_type** | `str` | ระบุรูปแบบ: `"binary"` สำหรับ QUBO หรือ `"spin"` สำหรับ Ising | `"binary"` |
| **backend_name** | `str` | อุปกรณ์ quantum เป้าหมาย | `"ibm_fez"` |

#### แนวคิดสำคัญ
- **รูปแบบปัญหา**: เราใช้ `"binary"` เพราะตัวแปรของเราเป็น binary (0/1) ซึ่งแทนการกำหนดตลาด
- **การเลือก Backend**: เลือกระหว่าง QPU ที่มีให้ใช้งาน (เช่น `"ibm_fez"`) ตามความต้องการและ compute resource instance ของคุณ
- **โครงสร้าง QUBO**: dictionary ปัญหาของเราประกอบด้วย coefficients ที่แน่นอนจากการแปลงทางคณิตศาสตร์

#### ตัวเลือกขั้นสูง (optional)
Iskay มีความสามารถในการปรับแต่งผ่านพารามิเตอร์ optional แม้ค่าเริ่มต้นจะทำงานได้ดีสำหรับปัญหาส่วนใหญ่ แต่คุณสามารถปรับแต่งพฤติกรรมสำหรับความต้องการเฉพาะได้:

| พารามิเตอร์ | ประเภท | ค่าเริ่มต้น | คำอธิบาย |
|-----------|------|---------|-------------|
| **shots** | `int` | 10000 | การวัด quantum ต่อ iteration (มากขึ้น = แม่นยำขึ้น) |
| **num_iterations** | `int` | 10 | จำนวน iteration ของอัลกอริทึม (iteration มากขึ้นอาจปรับปรุงคุณภาพ solution) |
| **use_session** | `bool` | True | ใช้ IBM sessions เพื่อลดเวลาคิว |
| **seed_transpiler** | `int` | None | ตั้งค่าเพื่อให้ quantum circuit compilation ได้ผลลัพธ์ที่ reproducible |
| **direct_qubit_mapping** | `bool` | False | Map virtual qubits โดยตรงไปยัง physical qubits |
| **job_tags** | `List[str]` | None | tags กำหนดเองสำหรับการติดตาม job |
| **preprocessing_level** | `int` | 0 | ความเข้มของการ preprocessing ปัญหา (0-3) - ดูรายละเอียดด้านล่าง |
| **postprocessing_level** | `int` | 2 | ระดับการปรับแต่ง solution (0-2) - ดูรายละเอียดด้านล่าง |
| **transpilation_level** | `int` | 0 | จำนวนครั้งในการ optimization ของ Transpiler (0-5) - ดูรายละเอียดด้านล่าง |
| **transpile_only** | `bool` | False | วิเคราะห์ circuit optimization โดยไม่ต้องรัน execution เต็มรูปแบบ |

**Preprocessing Levels (0-3)**: สำคัญเป็นพิเศษสำหรับปัญหาขนาดใหญ่ที่ไม่สามารถพอดีกับ coherence times ของฮาร์ดแวร์ในปัจจุบัน Preprocessing level ที่สูงขึ้นจะได้ circuit depth ที่ตื้นกว่าโดยการประมาณใน problem transpilation:
- **Level 0**: แน่นอน circuit ยาวกว่า
- **Level 1**: สมดุลที่ดีระหว่างความแม่นยำและการประมาณ ตัด gate ที่มีมุมอยู่ใน percentile ต่ำสุด 10 เท่านั้น
- **Level 2**: การประมาณที่สูงขึ้นเล็กน้อย ตัด gate ที่มีมุมอยู่ใน percentile ต่ำสุด 20 และใช้ `approximation_degree=0.95` ใน transpilation
- **Level 3**: ระดับการประมาณสูงสุด ตัด gate ที่อยู่ใน percentile ต่ำสุด 30 และใช้ `approximation_degree=0.90` ใน transpilation

**Transpilation Levels (0-5)**: ควบคุมจำนวนครั้งในการ optimization ของ Transpiler ขั้นสูงสำหรับ quantum circuit compilation ซึ่งอาจเพิ่ม classical overhead และในบางกรณีอาจไม่เปลี่ยน circuit depth ค่าเริ่มต้น `2` โดยทั่วไปให้ Circuit ที่เล็กที่สุดและค่อนข้างเร็ว
- **Level 0**: Optimization ของ DCQO circuit ที่ decompose แล้ว (layout, routing, scheduling)
- **Level 1**: Optimization ของ `PauliEvolutionGate` แล้วจึง DCQO circuit ที่ decompose แล้ว (max_trials=10)
- **Level 2**: Optimization ของ `PauliEvolutionGate` แล้วจึง DCQO circuit ที่ decompose แล้ว (max_trials=15)
- **Level 3**: Optimization ของ `PauliEvolutionGate` แล้วจึง DCQO circuit ที่ decompose แล้ว (max_trials=20)
- **Level 4**: Optimization ของ `PauliEvolutionGate` แล้วจึง DCQO circuit ที่ decompose แล้ว (max_trials=25)
- **Level 5**: Optimization ของ `PauliEvolutionGate` แล้วจึง DCQO circuit ที่ decompose แล้ว (max_trials=50)

**Postprocessing Levels (0-2)**: ควบคุมปริมาณการ optimization แบบ classical เพื่อชดเชยข้อผิดพลาด bit-flip ด้วยจำนวน greedy pass ของ local search ที่แตกต่างกัน:
- **Level 0**: 1 pass
- **Level 1**: 2 passes
- **Level 2**: 3 passes

**โหมด transpile-only**: ปัจจุบันพร้อมใช้งานสำหรับผู้ที่ต้องการวิเคราะห์ circuit optimization โดยไม่ต้องรัน quantum algorithm execution เต็มรูปแบบ

#### ตัวอย่างการ configuration แบบกำหนดเอง
นี่คือวิธีที่คุณอาจ configure Iskay ด้วยการตั้งค่าที่แตกต่างกัน:

In [ ]:
# Submit the optimization job
print("Submitting optimization job to Kipu Quantum...")
print(
    f"Problem size: {A.shape[1]} variables, {len(iskay_input['problem'])} terms"
)
print(
    "Algorithm: bf-DCQO (bias-field digitized counterdiabatic quantum optimization)"
)

job = iskay_solver.run(**iskay_input)

print("\nJob successfully submitted!")
print(f"Job ID: {job.job_id}")
print("Optimization in progress...")
print(
    f"The bf-DCQO algorithm will efficiently explore {2**A.shape[1]:,} possible assignments"
)

สำหรับ tutorial นี้ เราจะใช้ค่า default ส่วนใหญ่ และจะเปลี่ยนแค่จำนวน bias-field iterations:

In [ ]:
# Check job status
print(f"Job status: {job.status()}")

## ขั้นตอนที่ 3: รันด้วย Qiskit primitives
ตอนนี้เราส่งปัญหาไปรันบน IBM Quantum hardware แล้ว อัลกอริทึม bf-DCQO จะทำดังนี้:
1. สร้าง Circuit quantum แบบตื้นที่มี counterdiabatic terms
2. รันประมาณ 10 รอบด้วย bias-field optimization
3. ทำ post-processing แบบ classical ด้วย local search
4. คืนค่าการแบ่งตลาดที่ดีที่สุด

In [ ]:
# Wait for job completion
while True:
    status = job.status()
    print(
        f"Waiting for job {job.job_id} to complete... (status: {status})",
        end="\r",
        flush=True,
    )
    if status in ["DONE", "CANCELED", "ERROR"]:
        print(
            f"\nJob {job.job_id} completed with status: {status}" + " " * 20
        )
        break
    time.sleep(30)

# Retrieve the optimization results
result = job.result()
print("\nOptimization complete!")

### ติดตามสถานะ job
ตรวจสอบสถานะปัจจุบันของ optimization job ได้เลย สถานะที่เป็นไปได้มีดังนี้:
- `QUEUED`: Job รอคิวอยู่
- `RUNNING`: Job กำลังรันบน quantum hardware
- `DONE`: Job เสร็จสมบูรณ์
- `CANCELED`: Job ถูกยกเลิก
- `ERROR`: Job เจอข้อผิดพลาด

In [ ]:
# Display the optimization results
print("Optimization Results")
print("=" * 50)
print(f"Problem Type: {result['prob_type']}")
print("\nSolution Info:")
print(f"  Bitstring: {result['solution_info']['bitstring']}")
print(f"  Cost: {result['solution_info']['cost']}")
print("\nSolution (first 10 variables):")
for i, (var, val) in enumerate(list(result["solution"].items())[:10]):
    print(f"  {var}: {val}")
print("  ...")

### รอให้เสร็จ
Cell นี้จะหยุดรอจนกว่า job จะเสร็จ กระบวนการ optimization ประกอบด้วย:
- เวลาคิว (รอเข้าถึง quantum hardware)
- เวลารัน (รัน bf-DCQO algorithm ประมาณ 10 รอบ)
- เวลา post-processing (classical local search)

โดยทั่วไปใช้เวลาตั้งแต่ไม่กี่นาทีถึงหลายสิบนาที ขึ้นอยู่กับสภาพคิวในขณะนั้น

In [ ]:
def validate_solution(A, b, solution):
    """Validate market split solution."""
    x = np.array(solution)
    region_a = A @ x
    region_b = A @ (1 - x)
    violations = np.abs(region_a - b)

    return {
        "target": b,
        "region_a": region_a,
        "region_b": region_b,
        "violations": violations,
        "total_violation": np.sum(violations),
        "is_feasible": np.sum(violations) == 0,
        "region_a_markets": int(np.sum(x)),
        "region_b_markets": len(x) - int(np.sum(x)),
    }


# Convert bitstring to list of integers and validate
optimal_assignment = [
    int(bit) for bit in result["solution_info"]["bitstring"]
]
validation = validate_solution(A, b, optimal_assignment)

## ขั้นตอนที่ 4: Post-process และแปลงผลลัพธ์เป็นรูปแบบ classical ที่ต้องการ
ตอนนี้เรา post-process ผลลัพธ์จากการรัน quantum แล้ว ซึ่งรวมถึง:
- วิเคราะห์โครงสร้างของ solution
- ตรวจสอบความถูกต้องของ constraint
- เปรียบเทียบกับแนวทาง classical

### วิเคราะห์ผลลัพธ์
#### ทำความเข้าใจโครงสร้างผลลัพธ์
Iskay คืน result dictionary ที่ครบครันซึ่งประกอบด้วย:
- **`solution`**: Dictionary ที่ map ดัชนีตัวแปรไปยังค่าที่ดีที่สุด (0 หรือ 1)
- **`solution_info`**: ข้อมูลละเอียด ได้แก่:
  - `bitstring`: การมอบหมายที่ดีที่สุดในรูป binary string
  - `cost`: ค่าของ objective function (ควรเป็น 0 หากตอบสนอง constraint ได้สมบูรณ์)
  - `mapping`: วิธีที่ตำแหน่งใน bitstring map ไปยังตัวแปรของปัญหา
  - `seed_transpiler`: seed ที่ใช้เพื่อให้ผลลัพธ์ reproducible
- **`prob_type`**: ว่า solution อยู่ในรูป binary หรือ spin

มาดู solution ที่ quantum optimizer คืนมากัน

In [ ]:
print("Solution Validation")
print("=" * 50)
print(f"Feasible solution: {validation['is_feasible']}")
print(f"Total constraint violation: {validation['total_violation']}")

print("\nSales Analysis (Target vs Actual):")
for i, (target, actual_a, actual_b) in enumerate(
    zip(validation["target"], validation["region_a"], validation["region_b"])
):
    violation_a = abs(actual_a - target)
    violation_b = abs(actual_b - target)
    print(f"  Product {i+1}:")
    print(f"    Target: {target}")
    print(f"    Region A: {actual_a} (violation: {violation_a})")
    print(f"    Region B: {actual_b} (violation: {violation_b})")

print("\nMarket Distribution:")
print(f"  Region A: {validation['region_a_markets']} markets")
print(f"  Region B: {validation['region_b_markets']} markets")

#### ตรวจสอบความถูกต้องของ solution
ตอนนี้ตรวจสอบว่า quantum solution ตอบสนอง Market Split constraints หรือไม่ กระบวนการตรวจสอบจะดูสิ่งต่อไปนี้:

**การละเมิด constraint คืออะไร?**
- สำหรับแต่ละผลิตภัณฑ์ $i$ เราคำนวณยอดขายจริงในภูมิภาค A: $(Ax)_i$
- เราเปรียบเทียบกับยอดขายเป้าหมาย $b_i$
- **การละเมิด** คือค่าต่างในค่าสัมบูรณ์: $|(Ax)_i - b_i|$
- **solution ที่ feasible** มีการละเมิดเป็นศูนย์สำหรับทุกผลิตภัณฑ์

**สิ่งที่คาดหวัง:**
- **กรณีที่ดีที่สุด**: การละเมิดรวม = 0 (ตอบสนอง constraint ได้สมบูรณ์ทุกข้อ)
  - ภูมิภาค A ได้รับผลิตภัณฑ์ 1 จำนวน 1002 หน่วย ผลิตภัณฑ์ 2 จำนวน 879 หน่วย และผลิตภัณฑ์ 3 จำนวน 1040 หน่วยพอดี
  - ภูมิภาค B ได้รับหน่วยที่เหลือ (1002, 879 และ 1040 ตามลำดับเช่นกัน)
- **กรณีที่ดี**: การละเมิดรวมมีค่าน้อย (near-optimal solution)
- **กรณีไม่ดี**: การละเมิดมากแสดงว่า solution ไม่ตอบสนองความต้องการทางธุรกิจ

ฟังก์ชันตรวจสอบจะคำนวณ:
1. ยอดขายจริงต่อผลิตภัณฑ์ในแต่ละภูมิภาค
2. การละเมิด constraint สำหรับแต่ละผลิตภัณฑ์
3. การกระจายตลาดระหว่างภูมิภาค